# Replicar mBERT — Correcciones de auditoría (Cristian)

Libreta única para Google Colab. Replica las evaluaciones T-02 a T-06 del proyecto mBERT.

**Antes de ejecutar:** Runtime → Cambiar tipo de entorno → **GPU (T4)**.

| Modo | Descripción |
|------|-------------|
| **Verificación** | Sube `mbert-sv-audit.zip` → F1 oficial **0.7777** |
| **Reentrenamiento** | Fine-tune desde cero (~2–4 h) → F1 puede variar |

> ⚠️ Reentrenar en Colab **no garantiza** F1 = 0.7777. Esa cifra se reproduce con el checkpoint T-02.

## 1. Instalar dependencias

In [ ]:
!pip install -q -r requirements-colab.txt
!pip install -q tabulate

## 2. Extraer proyecto

Sube `mbert-cristian-colab.zip`.

In [ ]:
from google.colab import files
import shutil
import zipfile
from pathlib import Path

PROJECT_NAME = 'metricas-proyecto-ml'
ZIP_DEST = Path('/content/mbert-cristian-colab.zip')
ROOT = Path('/content') / PROJECT_NAME

uploaded = files.upload()
if uploaded:
    zip_name = next(iter(uploaded))
    ZIP_DEST.write_bytes(uploaded[zip_name])
elif ZIP_DEST.exists():
    print(f'Reutilizando: {ZIP_DEST}')
else:
    raise FileNotFoundError('Sube mbert-cristian-colab.zip')

if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

with zipfile.ZipFile(ZIP_DEST) as zf:
    for info in zf.infolist():
        name = info.filename.replace('\\', '/')
        if not name or name.endswith('/'):
            continue
        target = ROOT / name
        if PROJECT_NAME in name:
            rel = name.split(PROJECT_NAME + '/', 1)[-1]
            target = ROOT / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(info) as src, open(target, 'wb') as dst:
            dst.write(src.read())

import sys
sys.path.insert(0, str(ROOT / 'src'))
%cd {ROOT}
print('Proyecto en:', ROOT)
print('Train:', (ROOT / 'data/processed/train.csv').exists())

## 3. Cargar checkpoint (opción recomendada)

Sube `mbert-sv-audit.zip`. Si omites este paso, se reentrenará en la sección 4.

In [ ]:
import zipfile
from pathlib import Path

MODEL_DIR = ROOT / 'models' / 'mbert-sv'
USE_PRETRAINED = False

audit_zip = Path('/content/mbert-sv-audit.zip')
uploaded_audit = files.upload()
if uploaded_audit:
    name = next(iter(uploaded_audit))
    audit_zip.write_bytes(uploaded_audit[name])

if audit_zip.exists():
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(audit_zip) as zf:
        for info in zf.infolist():
            if info.filename.endswith('/'):
                continue
            parts = info.filename.replace('\\', '/').split('/')
            fname = parts[-1]
            if fname in ('model.safetensors', 'config.json', 'tokenizer.json',
                         'tokenizer_config.json', 'inference_contract.json'):
                target = MODEL_DIR / fname
                with zf.open(info) as src, open(target, 'wb') as dst:
                    dst.write(src.read())
    USE_PRETRAINED = MODEL_DIR.joinpath('model.safetensors').is_file()
    print('Checkpoint cargado:', USE_PRETRAINED)
else:
    print('Sin audit ZIP → se reentrenará en la siguiente sección.')

## 4. Fine-tuning mBERT (solo si no hay checkpoint)

In [ ]:
if not USE_PRETRAINED:
    from discurso_odio.data.loaders import load_split
    from discurso_odio.models.mbert import train_mbert, get_device
    from discurso_odio.utils.seed import set_seed
    from discurso_odio.config import load_settings

    settings = load_settings()
    set_seed(settings['project']['seed'])
    print('Dispositivo:', get_device())
    train_df = load_split('train')
    val_df = load_split('val')
    print(f'Train: {len(train_df)} | Val: {len(val_df)}')
    train_mbert(train_df, val_df, output_dir=MODEL_DIR)
    print('Entrenamiento completado.')
else:
    print('Omitiendo entrenamiento (checkpoint T-02 cargado).')

## 5. Evaluación oficial (T-04) — test n=460

In [ ]:
from discurso_odio.evaluation.official_mbert import evaluate_mbert_official
from discurso_odio.models.mbert_artifacts import OFFICIAL_F1_MACRO
from discurso_odio.data.loaders import load_split
from discurso_odio.models.mbert import predict
from discurso_odio.evaluation.metrics import compute_all_metrics

try:
    result = evaluate_mbert_official(model_dir=MODEL_DIR)
    f1 = result['official_f1_macro']
    print('Evaluación oficial T-04 completada.')
except ValueError as exc:
    print(f'Fuera de tolerancia oficial ({exc}); mostrando métricas del modelo actual.')
    test = load_split('test')
    y_pred, y_prob = predict(test['texto_modelo'].tolist(), MODEL_DIR)
    metrics = compute_all_metrics(test['label'].values, y_pred, y_prob)
    f1 = float(metrics['f1_macro'])

print(f'F1-macro test: {f1:.4f}')
print(f'Oficial esperado (T-02): {OFFICIAL_F1_MACRO:.4f}')
print(f'Delta: {abs(f1 - OFFICIAL_F1_MACRO):.4f}')
if abs(f1 - OFFICIAL_F1_MACRO) <= 1e-3:
    print('OK: dentro de tolerancia')
else:
    print('Esperado si reentrenaste en Colab (no usa checkpoint T-02).')

## 6. Cortes y McNemar (T-05 / T-03)

In [ ]:
import pandas as pd
from discurso_odio.evaluation.metrics import mcnemar_test
from discurso_odio.evaluation.predictions import load_mbert_predictions, load_xlmr_predictions
from discurso_odio.paths import REPORTS_METRICS, TEST_CSV
from discurso_odio.data.loaders import load_split

test = load_split('test')
slice_df = pd.read_csv(REPORTS_METRICS / 'slice_metrics.csv')
mbert_rows = slice_df[slice_df['model'] == 'mbert'] if 'model' in slice_df.columns else slice_df
print('Cortes (mBERT):')
print(mbert_rows[['slice', 'n', 'f1_macro']].head(8).to_string(index=False))

y_pred_mbert, _ = load_mbert_predictions(test)
if y_pred_mbert is None:
    preds = pd.read_csv(REPORTS_METRICS / 'mbert_test_predictions.csv')
    merged = test.merge(preds, left_on='ID', right_on='id')
    y_pred_mbert = merged['y_pred_mbert'].values

y_pred_xlmr, _ = load_xlmr_predictions(test)
if y_pred_xlmr is not None:
    stat = mcnemar_test(test['label'].values, y_pred_mbert, y_pred_xlmr)
    print(f"\nMcNemar mBERT vs XLM-R: p={stat['p_value']:.6f}, b={stat['b']}, c={stat['c']}")
else:
    print('xlmr_test_predictions.csv no disponible.')

## 7. Batería adversarial (T-06)

In [ ]:
from discurso_odio.evaluation.adversarial import evaluate_adversarial_battery

adv = evaluate_adversarial_battery(model_dir=MODEL_DIR)
print(f"Adversarial global: {adv['global_correct']}/{adv['n_cases']}")
print(f"F1 diagnóstico: {adv['f1_macro_diagnostic']:.4f}")
print(f"Mitigación requerida: {adv['mitigation_required']}")
print(f"FPR gap dialectal: {adv['fpr_dialectal_gap']:.4f}")

## 8. Resumen de correcciones

| Tarea | Resultado esperado |
|-------|-------------------|
| T-01 | Baseline mayoritaria = 0.128 |
| T-02 | Checkpoint verificable (F1 0.7777) |
| T-03 | McNemar vs XLM-R: p ≈ 0.200 |
| T-04 | F1-macro oficial = 0.7777 |
| T-05 | Cortes en slice_metrics.csv |
| T-06 | Adversarial 45/75; B2 **no adoptada** |
